<a href="https://colab.research.google.com/github/AnNguyenGia/CIFAR-10/blob/main/Cifar_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import PIL
import torchvision.models as models
import torch.nn as nn
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim

In [28]:
transform = transforms.Compose([
    transforms.Resize((224, 224)), # Scales height and width to 224x224
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

#import datasets

In [ ]:
df_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
df_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:02<00:00, 65.4MB/s]


In [ ]:
classes = df_train.classes

# CNN strategy

In [ ]:
EPOCHS = 15
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 5, padding=2) # 32x32 -> 32x32 -> Pool -> 16x16
        self.conv2 = nn.Conv2d(16, 32, 5, padding=2) # 16x16 -> 16x16 -> Pool -> 8x8
        self.conv3 = nn.Conv2d(32, 64, 5, padding=2) # 8x8 -> 8x8 -> Pool -> 4x4

        self.pool = nn.MaxPool2d(2, 2)
        self.fla = nn.Flatten()


        self.fc1 = nn.Linear(in_features=50176, out_features=1000)
        self.fc2 = nn.Linear(in_features=1000, out_features=120)
        self.fc3 = nn.Linear(in_features=120, out_features=84)
        self.fc4 = nn.Linear(in_features=84, out_features=10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.fla(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)

        return x



In [ ]:
from tqdm.auto import tqdm
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang train trên thiết bị: {device}")

model = CNN().to(device)

optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
trainloader = DataLoader(df_train, batch_size=32, shuffle=True)

patience = 3
epochs_no_improve = 0
best_loss = float('inf')

for epoch in range(EPOCHS):
    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}")

    epoch_loss = 0

    for image, label in pbar:
        # === BƯỚC 3: ĐƯA DATA LÊN CÙNG GPU VỚI MODEL ===
        image = image.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = F.cross_entropy(output, label)
        loss.backward()
        optimizer.step()

        pbar.set_postfix(loss=loss.item())
        epoch_loss += loss.item()

    # Tính trung bình loss và kiểm tra Early Stopping
    avg_loss = epoch_loss / len(trainloader)

    if avg_loss < best_loss:
        best_loss = avg_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"\nLoss không giảm. Cảnh báo: {epochs_no_improve}/{patience}")

        if epochs_no_improve >= patience:
            print(f"🚨 Kích hoạt Early Stopping! Dừng train tại Epoch {epoch+1}.")
            break

Đang train trên thiết bị: cpu


Epoch 1:   0%|          | 0/1563 [00:00<?, ?it/s]

In [ ]:

model.eval()

test_loss = 0.0
correct = 0
total = 0

# 2. Tắt tính toán đạo hàm (vô cùng quan trọng để tiết kiệm RAM/GPU)
with torch.no_grad():
    for image, label in df_test:
        # Chuẩn bị dữ liệu (unsqueeze để giả lập batch size = 1)
        image = image.unsqueeze(0).to(device)

        # Đưa label về dạng tensor đúng thiết bị
        if not isinstance(label, torch.Tensor):
            target = torch.tensor([label]).to(device)
        else:
            target = label.unsqueeze(0).to(device)

        # 3. Chạy model (Forward pass)
        output = model(image)

        # 4. Tính loss trên tập test
        loss = criterion(output, target)
        test_loss += loss.item()

        # 5. Tính số câu trả lời đúng
        # Lấy class có giá trị logit lớn nhất
        predicted = torch.argmax(output, dim=1)
        if predicted.item() == label:
            correct += 1

        total += 1

# --- TÍNH TOÁN KẾT QUẢ CUỐI CÙNG ---
avg_test_loss = test_loss / total
accuracy = 100 * correct / total

print("-" * 30)
print(f"KẾT QUẢ TRÊN TẬP TEST:")
print(f"Loss trung bình: {avg_test_loss:.4f}")
print(f"Độ chính xác (Accuracy): {accuracy:.2f}% ({correct}/{total} ảnh)")
print("-" * 30)

tensor([[ -2.4805, -16.3471,   0.8312,  24.2464,  -5.5784,  19.9826, -12.6956,
          -0.5310,   0.9926, -17.1356]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

#Transfer Learning strategy (VGGNet)

In [ ]:
vggnet = models.vgg16(pretrained=True)
for param in vggnet.features.parameters():
    param.requires_grad = False
in_features = model.classifier[6].in_features
vggnet.classifier = nn.Sequential(
    nn.Linear(in_features=in_features, out_features=4096, bias=True),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5, inplace=False),
    nn.Linear(in_features=4096, out_features=4096, bias=True),
)